# Deriving curvilinear $\nabla$ from a coordinate mapping (vibe 000069)

The point of vibe 000069: hand a chart its coordinate **mapping** and let the
library derive everything else — the moving frame, the metric, and the
differential operators — with no hand-supplied scale factors or Christoffel
symbols.

This notebook walks the **cylindrical** chart from its mapping
$x = r\cos\theta,\; y = r\sin\theta,\; z = z$ all the way to the textbook
operators, then shows the same machinery handle the **spherical** chart.

0. The mapping $\;\to\; R$
1. Tangent basis $g_i = \partial R/\partial q^i$ &nbsp; (M1 scalar fields + M2 differentiation)
2. Metric & scale factors $g_{ij},\; h_i = \sqrt{g_{ii}}$ &nbsp; (M3 simplifier + M4)
3. Physical frame $e_i = g_i/h_i$ — a real tender `Basis` &nbsp; (M4)
4. How the frame turns $\partial_j e_i = \sum_k \gamma^k_{ij}\, e_k$ &nbsp; (M5)
5. Operators $\nabla f,\; \operatorname{div} v,\; \Delta f,\; \operatorname{rot} v$ &nbsp; (M6)
6. The same machinery, spherical

In [ ]:
import tender
import tender.basis as tb
import tender.chart as tc
import tender.derivation as td
from IPython.display import Math, display

ctx = tender.Context()


def disp(*exprs):
    for e in exprs:
        display(Math(e.latex()))


def is_zero(e):
    return td.algebraic_eq(e, tender.scalar(0, ctx=ctx))


def disp_field(components, names, lhs=None):
    "Display a physical vector field sum_i c_i e_i, skipping zero components."
    parts = []
    for comp, name in zip(components, names):
        if is_zero(comp):
            continue
        s = comp.latex()
        vec = rf"\mathbf{{e}}_{{{name}}}"
        if s == "1":
            parts.append(vec)
        elif s == "-1":
            parts.append("-" + vec)
        else:
            parts.append(rf"\left({s}\right)\,{vec}")
    body = " + ".join(parts).replace("+ -", "-") or "0"
    display(Math((lhs + " = " if lhs else "") + body))

## 0. The mapping $\to$ the position vector $R$

A chart targets an orthonormal Cartesian reference frame (here WCS $i, j, k$) and
is given its coordinate variables and the Cartesian components $x^a = f^a(q)$.
The single domain fact $r \ge 0$ (`nonneg=True`) is all the simplifier needs
later for $\sqrt{r^2} = r$.

In [ ]:
cart = tb.wcs(ctx)
r = tender.coordinate("r", chart_id=1, slot=0, nonneg=True, ctx=ctx)
th = tender.coordinate(r"\theta", chart_id=1, slot=1, ctx=ctx)
z = tender.coordinate("z", chart_id=1, slot=2, ctx=ctx)

cyl = tc.CoordinateChart(
    cart,
    [r, th, z],
    [r * tender.cos(th), r * tender.sin(th), z],  # x, y, z
)
names = ["r", r"\theta", "z"]

disp(cyl.radius_vector())  # R = r cos(theta) i + r sin(theta) j + z k

## 1. Tangent (holonomic) basis $g_i = \partial R/\partial q^i$

Differentiating $R$ coordinate-by-coordinate is the scalar-field differentiation
engine at work: $\partial_\theta(r\cos\theta) = -r\sin\theta$, and the constant
reference vectors $i, j, k$ differentiate to $0$.

In [ ]:
g = [cyl.tangent_vector(i) for i in range(3)]
disp(*g)  # g_r, g_theta, g_z

## 2. Metric and scale factors $g_{ij},\; h_i = \sqrt{g_{ii}}$

$g_{\theta\theta} = r^2$ needs $\cos^2\theta + \sin^2\theta \to 1$, and
$h_\theta = \sqrt{r^2} \to r$ needs $r \ge 0$ — both folds are the targeted
scalar simplifier, fired automatically.  The off-diagonal $g_{r\theta} = 0$, so
the frame is orthogonal.

In [ ]:
disp(
    cyl.metric_component(0, 0),  # g_rr = 1
    cyl.metric_component(1, 1),  # g_theta theta = r^2
    cyl.metric_component(0, 1),  # g_r theta = 0
)
h = [cyl.scale_factor(i) for i in range(3)]
disp(*h)  # h_r = 1, h_theta = r, h_z = 1

## 3. Physical orthonormal frame $e_i = g_i / h_i$

Normalising the tangent vectors gives the physical frame — and it is returned as
a genuine tender `Basis` (orthonormal, carrying the coordinate letters), so it
plugs into the rest of the library.

In [ ]:
frame = cyl.physical_basis()
disp(*[frame.basis(i) for i in range(3)])  # e_r, e_theta, e_z
print("dim:", frame.dim, " orthonormal:", frame.is_orthonormal)

## 4. How the frame turns: $\partial_j e_i = \sum_k \gamma^k_{ij}\, e_k$

The connection (rotation) coefficients re-express each $\partial e$ back in the
local frame.  These are **derived, not tabulated** — and they are exactly what
the operators below consume.  The classic polar/cylindrical formulas
$\partial_\theta e_r = e_\theta$ and $\partial_\theta e_\theta = -e_r$ fall out.

In [ ]:
disp_field(cyl.connection_coefficients(0, 1), names, r"\partial_\theta e_r")  # = e_theta
disp_field(cyl.connection_coefficients(1, 1), names, r"\partial_\theta e_\theta")  # = -e_r
disp_field(cyl.connection_coefficients(0, 0), names, r"\partial_r e_r")  # = 0

## 5. Differential operators $\;\nabla = \sum_i \frac{1}{h_i}\, e_i\, \partial_{q^i}$

Each operator is $\nabla$ applied by the formal rule and returns an **invariant
tensor** — no component tuples.  `grad` raises the rank by one, so for the
position vector $\nabla R = \sum_i e_i \otimes e_i = I$, the identity tensor; for
a scalar it surfaces the $1/h$ factors, i.e.
$\nabla = e_r\,\partial_r + \frac1r e_\theta\,\partial_\theta + e_z\,\partial_z$.
`div`, $\Delta$ and `rot` follow on vector fields built straight from the frame
and match the textbook formulas: $\nabla\cdot(r\,e_r) = 2$, $\Delta(r^2) = 4$,
$\nabla\times(r\,e_\theta) = 2\,e_z$ (uniform vorticity).

In [ ]:
frame = cyl.physical_basis()
e = [frame.basis(i) for i in range(3)]
R = cyl.radius_vector()


def simp(x):  # distribute (x) over + and simplify, for clean display
    return td.simplify_scalars(td.expand_products(x))


disp(simp(cyl.grad(R)))     # grad R = I  (i i + j j + k k)
disp(simp(cyl.grad(th)))    # grad theta = (1/r) e_theta
disp(simp(cyl.grad(r**2)))  # grad r^2 = 2r e_r
disp(cyl.div(r * e[0]))  # div(r e_r) = 2
disp(cyl.laplacian(r**2))       # Laplacian(r^2) = 4
disp(simp(cyl.rot(r * e[1])))   # rot(r e_theta) = 2 e_z

## 6. The same machinery, spherical

$x = r\sin\theta\cos\phi,\; y = r\sin\theta\sin\phi,\; z = r\cos\theta$.  Nothing
changes but the mapping: the two-trig-square Pythagorean fold gives
$h_\phi = \sqrt{r^2\sin^2\theta} = r\sin\theta$, the connection yields
$\partial_\phi e_\phi = -\sin\theta\, e_r - \cos\theta\, e_\theta$, and
$\Delta(r^2) = 6$.

In [ ]:
rs = tender.coordinate("r", chart_id=2, slot=0, nonneg=True, ctx=ctx)
ts = tender.coordinate(r"\theta", chart_id=2, slot=1, ctx=ctx)
ps = tender.coordinate(r"\phi", chart_id=2, slot=2, ctx=ctx)
sph = tc.CoordinateChart(
    tb.wcs(ctx),
    [rs, ts, ps],
    [
        rs * tender.sin(ts) * tender.cos(ps),
        rs * tender.sin(ts) * tender.sin(ps),
        rs * tender.cos(ts),
    ],
)
sph_names = ["r", r"\theta", r"\phi"]

disp(sph.scale_factor(2))  # h_phi = r sin(theta)
disp_field(sph.connection_coefficients(2, 2), sph_names, r"\partial_\phi e_\phi")
disp(sph.laplacian(rs**2))  # Laplacian(r^2) = 6